<a href="https://colab.research.google.com/github/yohyama0216/go-tools/blob/main/GPUCrossword.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import torch
import numpy as np
import time

# --- 1. 環境設定 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

chars = "アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワヲン" \
        "ガギグゲゴザジズゼゾダヂヅデドバビブベボパピプペポー"
char_to_int = {c: i for i, c in enumerate(chars)}
int_to_char = {i: c for i, c in enumerate(chars)}

# --- 2. 単語リストの統合と浄化 ---
plant_list_raw = [
    "アオキ", "アセビ", "アジサイ", "イチゴ", "ウメ", "エノキ", "カキ", "カエデ", "クリ", "クヌギ", "サクラ", "サカキ", "スギ", "スミレ", "ツツジ", "ツバキ", "トマト", "ナス", "ハス", "バラ", "ヒノキ", "マツ", "モモ", "ユリ", "リンゴ", "ワサビ", "サルスベリ", "ヒマワリ", "レンゲソウ", "ラベンダー", "シンリンヨク", "コウヨウジュ", "シンヨウジュ", "カンヨウショクブツ", "スイショウギク",
    "ハナバタケ", "ヤサイバタケ", "カジエン", "ミドリンリ", "ボタニカル", "オーガニック", "プランター", "ガーデニング", "テラリウム", "アクアリウム",
    "タネ", "クキ", "ツボミ", "カブ", "ツル", "サボテン", "セリ", "フキ", "ワラビ", "ゼンマイ", "ポプラ", "ハゼノキ", "シラカバ", "オリーブ", "プラタナス", "ユーカリ", "ベニバナ", "シロツメ",
    "アロエベラ", "サボテンカ", "ヒマワリ油", "カボチャ種", "レモングラス", "カモミール", "マリーゴールド", "プランテーション", "バイオテクノ", "エコロジー",
    "ハナ", "シダ", "コケ", "クコ", "マメ", "イモ", "ウリ", "ニラ", "シバ", "モミジ", "スイカ", "アケビ", "コナラ", "ヤツデ", "アンズ", "カツラ", "シイノ", "タケノ", "ナズナ", "ボタン", "ムギノ", "レモン", "イグサ", "オクラ", "クルミ", "クワノ", "シオン", "シソノ", "セリナ", "タラノ", "ニレノ", "ハスの", "ブナノ", "フヨウ",
    "アサガオ", "パンジー", "コスモス", "キャベツ", "カボチャ", "キュウリ", "スイセン", "イヌマキ", "エゴノキ", "ガマズミ", "クチナシ", "コブシ", "サザンカ", "ネジキ", "ネムノキ", "ヤマブキ", "アセビノ", "アベマキ", "アマチャ", "アラカシ", "イヌブナ", "イチジク", "エニシダ", "カカオノ", "カマツカ", "キハダノ", "キブシノ", "クマシデ", "クマザサ", "ケヤキノ", "シラカシ", "シロダモ", "シロモジ", "ススキノ", "センダン", "ダイコン", "タケノコ", "タマネギ", "ニシキギ", "ニレノキ", "ハスノハ", "バナナノ", "ヒイラギ", "ブドウノ", "ホウノキ", "ミカンノ", "ミズキノ", "ムクノキ", "モクレン", "ヤナギノ", "ユズノキ", "ロウバイ",
    "チューリップ", "ブロッコリー", "ジャガイモ", "サツマイモ", "アズキナシ", "イロハカエデ", "ウグイスカ", "ウラジロガ", "カレンデュ", "キンモクセ", "ギンモクセ", "クロガネモ", "コシアブラ", "サトイモノ", "サンショウ", "シクラメン", "シラカンバ", "スイセンノ", "タマアジサ", "チャノキノ", "トチノキノ", "トドマツノ", "ナナカマド", "ナツツバキ", "ネズミモチ", "ノリウツギ", "ハクモクレ", "ハナミズキ", "ヒガンバナ", "ヒサカキノ", "ピラカンサ", "ベニバナア", "マタタビノ", "マテバシイ", "ミツバツツ", "ムラサキシ", "メグスリノ", "メタセコイ", "モチノキノ", "モチツツジ", "ヤマザクラ", "ヤマツツジ", "ヤマボウシ", "ユリノキノ", "ラベンダー", "ローズマリ",
    "キンモクセイ", "ギンモクセイ", "ベニバナアセ", "モチツツジノ", "ヤマザクラノ", "カモミールノ", "シラカンバノ", "ヒガンバナノ", "ムラサキシキ", "メタセコイア", "プランターノ", "ボタニカルナ", "テラリウムノ", "ハナバタケノ", "ヤサイバタケ", "バイオテクノ", "エコロジーナ",
    "アマ", "イキ", "ウシ", "エマ", "オカ", "カワ", "クニ", "サケ", "シカ", "ソラ", "タコ", "チカ", "ツユ", "ネコ", "ハレ", "ホシ", "マキ", "ムシ", "ユキ", "ヨル",
    "メダカ", "スズメ", "カモメ", "イワシ", "クジラ", "メロン", "ピアノ", "ダンス", "カメラ", "テスト", "サイン", "ライト", "ルビー", "インク", "カヌー", "ヨット", "ハワイ", "パリノ", "アジア",
    "ライオン", "キリンノ", "シマウマ", "ペンギン", "メダカノ", "サくらソ", "カーネシ", "デザート", "ソースノ", "ステーキ", "パスタノ", "ピザノハ", "コーラノ", "コーヒー", "ミルクノ", "ビールの",
    "アフリカノ", "アメリカノ", "イタリアノ", "フランスノ", "ブラジルノ", "オランダノ", "スペインノ", "ドイツノク", "インドネシ", "マレーシア", "カンボジア", "ベトナムノ", "メキシコノ", "エジプトノ", "トルコノキ", "ロシアノキ",
    "オーストラリア", "アルゼンチンノ", "フィリピンノキ", "シンガポールノ", "スイスアルプス", "スカンジナビア", "カリフォルニア", "カサブランカノ", "エーゲカイノハ",
    "アイス", "エプロン", "オート", "ギター", "クイズ", "ケース", "コア", "サイト", "ジャム", "スキー", "スノボ", "タイツ", "ドア", "ノート", "ハイフ", "ピアノ",
    "ベース", "マスク", "ランチ", "ライト", "ラジオ", "レンズ", "ワーク", "ベスト", "ポスト", "マニア",
    "アプリ", "イカ", "エンジン", "ガラス", "クラス", "コイン", "コース", "コップ", "サイン", "サイト", "ジャム", "スキー", "スコア", "スター", "ステーキ", "ストア", "テスト", "データ", "ドラマ", "ノート", "バイク", "パス", "ピアノ",
    "ビデオ", "ビール", "ペン", "ボタン", "マスク", "ミシン", "モデル", "ランチ", "ライト", "ラジオ", "リボン", "レンズ", "ワイン", "ワーク", "パン", "ケーキ", "チョコ", "ピザ", "バナナ", "リンゴ",
    "エマキ", "アニマル", "イベント", "オレンジ", "カラオケ", "キャンプ", "クイズノ", "クラスノ", "ケーキノ", "サイトノ", "サラダノ", "ショップ", "スマホノ", "タイヤノ", "チョコノ", "テレビノ", "トースト", "ニュース", "ネクタイ", "パソコン", "ヒーター", "ブラシノ", "プロセス", "ポスター", "マフラー", "メガネノ", "メッセージ", "メニュー", "リズムノ", "ルールノ", "ロケット",
    "アドバンス", "オフィスノ", "カテゴリノ", "カレンダー", "キーボード", "サービスノ", "システムノ", "シャンプー", "スピードノ", "スポーツノ", "センターノ", "チェックノ", "デザインノ", "トラブルノ", "ネットワーク", "ハイキング", "ピクニック", "フロントノ", "プログラム", "プレゼント", "ボーナスノ", "マイホーム", "ミュージカ", "メンバーノ", "リラックス", "レンアイノ", "ワールドノ"
]
common_list = ["アイス", "カメラ", "ノート", "クラス", "サイン", "ライト", "テスト", "アプリ", "エンジン", "センター", "メニュー", "データ", "ユーザー", "モデル", "ソフト", "システム", "レベル", "ドラマ", "ゲーム", "ニュース", "チーム", "サービス", "アフリカ", "イタリア", "フランス", "ロンドン", "ピアノ", "ギター", "ダンス"]

# --- クリーニングと変換 ---
small_to_large = str.maketrans("ァィゥェォッャュョ", "アイウエオツヤユヨ")
allowed_chars = set(chars)

def clean_word_list(raw_list):
    cleaned = []
    for w in raw_list:
        w_conv = w.translate(small_to_large)
        if all(c in allowed_chars for c in w_conv) and not w_conv.startswith("ー"):
            cleaned.append(w_conv)
    return list(set(cleaned))

all_words = clean_word_list(plant_list_raw + common_list)

def build_word_tensors(word_list):
    tensors_by_length = {}
    for word in word_list:
        l = len(word)
        if l not in tensors_by_length: tensors_by_length[l] = []
        tensors_by_length[l].append([char_to_int[c] for c in word])
    for l in tensors_by_length:
        tensors_by_length[l] = torch.tensor(tensors_by_length[l], dtype=torch.int16).to(device)
    return tensors_by_length

word_tensors = build_word_tensors(all_words)

# --- 3. 盤面設定 ---
grid = np.array([
    [0, 1, 0, 0, 0, 1],
    [0, 0, 0, 1, 0, 0],
    [1, 0, 1, 0, 0, 1],
    [1, 0, 0, 0, 1, 0],
    [1, 0, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 0]
])

def get_slots_and_intersections(grid):
    slots = []
    h, w = grid.shape
    for r in range(h):
        c = 0
        while c < w:
            if grid[r,c] == 0:
                start = c
                while c < w and grid[r,c] == 0: c += 1
                if c - start >= 2: slots.append({'dir': 'H', 'len': c-start, 'coords': [(r, i) for i in range(start, c)]})
            else: c += 1
    for c in range(w):
        r = 0
        while r < h:
            if grid[r,c] == 0:
                start = r
                while r < h and grid[r,c] == 0: r += 1
                if r - start >= 2: slots.append({'dir': 'V', 'len': r-start, 'coords': [(i, c) for i in range(start, r)]})
            else: r += 1
    intersections = []
    for i, s1 in enumerate(slots):
        for j, s2 in enumerate(slots):
            if i < j and s1['dir'] != s2['dir']:
                common = set(s1['coords']) & set(s2['coords'])
                for coord in common:
                    intersections.append((i, s1['coords'].index(coord), j, s2['coords'].index(coord)))
    return slots, intersections

slots, intersections = get_slots_and_intersections(grid)

# --- 4. 改良版ソルバー ---
class RobustSolver:
    def __init__(self, slots, intersections, word_tensors):
        self.slots = slots
        self.intersections = intersections
        self.word_tensors = word_tensors
        self.masks = []
        for i, s in enumerate(slots):
            if s['len'] not in word_tensors:
                self.masks.append(torch.tensor([], dtype=torch.bool, device=device))
            else:
                self.masks.append(torch.ones(word_tensors[s['len']].shape[0], dtype=torch.bool, device=device))
        self.solution = [None] * len(slots)
        self.step_count = 0
        # 重複禁止用：使用済み単語の内容（タプル）を保存するセット
        self.used_words = set()

    def solve(self, depth=0):
        self.step_count += 1
        unsolved = [i for i, s in enumerate(self.solution) if s is None]
        if not unsolved: return True

        curr_idx = min(unsolved, key=lambda i: self.masks[i].sum().item())
        cand_count = self.masks[curr_idx].sum().item()

        if cand_count == 0: return False

        candidates = torch.where(self.masks[curr_idx])[0]
        candidates = candidates[torch.randperm(len(candidates))]

        original_masks = [m.clone() for m in self.masks]

        for cand_idx in candidates:
            idx = cand_idx.item()
            # 単語の内容を取得して重複チェック
            word_tuple = tuple(self.word_tensors[self.slots[curr_idx]['len']][idx].tolist())

            if word_tuple in self.used_words:
                continue

            self.solution[curr_idx] = idx
            self.used_words.add(word_tuple)

            self.masks[curr_idx][:] = False
            self.masks[curr_idx][idx] = True

            if self.propagate(curr_idx):
                if self.solve(depth + 1): return True

            # バックトラック
            self.used_words.remove(word_tuple)
            self.solution[curr_idx] = None
            for i in range(len(self.masks)): self.masks[i][:] = original_masks[i]

        return False

    def propagate(self, fixed_idx):
        for s1, p1, s2, p2 in self.intersections:
            if s1 == fixed_idx or s2 == fixed_idx:
                target, t_pos, source, s_pos = (s2, p2, s1, p1) if s1 == fixed_idx else (s1, p1, s2, p2)
                valid_chars = self.word_tensors[self.slots[source]['len']][self.masks[source], s_pos].unique()
                self.masks[target] &= torch.isin(self.word_tensors[self.slots[target]['len']][:, t_pos], valid_chars)
                if not self.masks[target].any(): return False
        return True

# --- 実行 ---
solver = RobustSolver(slots, intersections, word_tensors)
print(f"探索開始... (有効単語数: {len(all_words)})")
start = time.time()
if solver.solve():
    print(f"\n生成成功！ 所要時間: {time.time()-start:.4f}s")
    res = np.full(grid.shape, "■", dtype=object)
    for i, s_idx in enumerate(solver.solution):
        w_ints = word_tensors[slots[i]['len']][s_idx].cpu().numpy()
        w_str = "".join([int_to_char[x] for x in w_ints])
        for char, (r, c) in zip(w_str, slots[i]['coords']): res[r,c] = char
    for row in res: print(" ".join(row))
else:
    print(f"\n解なし... 合計試行数: {solver.step_count}")

探索開始... (有効単語数: 415)

生成成功！ 所要時間: 0.0953s
タ ■ イ グ サ ■
ネ ジ キ ■ カ キ
■ ヤ ■ マ キ ■
■ ガ ラ ス ■ ド
■ イ ■ ク ジ ラ
イ モ ■ ■ ■ マ
